# 09.7 · How large language models work

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/07-how-llms-work/lesson.ipynb)

Read the [README](README.md) first and do the by-hand tables on paper. This notebook repeats them in code, then looks inside a real model.

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

## 1. A language model you can build by hand

Three sentences, count which word follows which. That is a **bigram model**: it predicts the next word from the previous one only.

In [1]:
from collections import Counter, defaultdict

sentences = ["the mug is red", "the mug is big", "the lamp is red"]
counts = defaultdict(Counter)                 # counts[word][next_word]
for s in sentences:
    words = s.split()
    for a, b in zip(words, words[1:]):
        counts[a][b] += 1

for word, nxt in counts.items():
    total = sum(nxt.values())
    print(f"after '{word}':", {w: f"{c}/{total}" for w, c in nxt.items()})

after 'the': {'mug': '2/3', 'lamp': '1/3'}
after 'mug': {'is': '2/2'}
after 'is': {'red': '2/3', 'big': '1/3'}
after 'lamp': {'is': '1/1'}


Exactly the README table: after *the*, *mug* 2/3 and *lamp* 1/3; after *is*, *red* 2/3 and *big* 1/3. Now let it write, always picking the most likely next word (this is called **greedy** decoding):

In [2]:
word, text = "the", ["the"]
while word in counts:
    word = counts[word].most_common(1)[0][0]
    text.append(word)
print(" ".join(text))

the mug is red


An LLM does the same job: given everything so far, score every possible next token, pick one, repeat. The differences are scale (billions of parameters instead of a table, trillions of words instead of three sentences) and context (it looks at thousands of previous tokens, not one word).

## 2. Tokens

Models do not read words. They read **tokens**: common words are one token, rarer words are split into pieces. Below, the tokenizer used by several OpenAI models (`cl100k_base`); every model family has its own, but the idea is the same.

In [3]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

for sentence in ["Thank you for your order.",
                 "ShopWave's WELCOME10 code gives 10% off.",
                 "Kenapa pesanan saya belum sampai?"]:      # Indonesian: "Why hasn't my order arrived?"
    ids = enc.encode(sentence)
    print(f"{len(sentence.split())} words → {len(ids)} tokens: {[enc.decode([i]) for i in ids]}")

5 words → 6 tokens: ['Thank', ' you', ' for', ' your', ' order', '.']
6 words → 13 tokens: ['Shop', 'Wave', "'s", ' W', 'ELCOME', '10', ' code', ' gives', ' ', '10', '%', ' off', '.']
5 words → 9 tokens: ['Ken', 'apa', ' pes', 'anan', ' saya', ' belum', ' samp', 'ai', '?']


Plain English is about one token per word. Brand names, codes and numbers split up (`WELCOME10` became three tokens). Languages that were rarer in the training data split even more: five Indonesian words became nine tokens. That matters because **you pay per token and the context window is measured in tokens**.

## 3. Looking inside a real model

Load the course helper and see which model you are using.

In [4]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [5]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


Ollama can show the probabilities a model gave to its top candidates for each token it wrote. (This cell talks to Ollama directly, because the other providers do not all expose this. With another provider it skips itself.)

In [6]:
import json, math, urllib.request

if course_llm.PROVIDER == "ollama":
    body = {"model": course_llm.MODEL, "stream": False, "logprobs": True, "top_logprobs": 5,
            "messages": [{"role": "user", "content": "Complete with one word: Thank you for your"}],
            "options": {"num_ctx": 8192, "num_predict": 3, "temperature": 0}}
    req = urllib.request.Request("http://localhost:11434/api/chat", json.dumps(body).encode())
    first = json.load(urllib.request.urlopen(req))["logprobs"][0]
    for cand in first["top_logprobs"]:
        print(f"{cand['token']!r:12} {math.exp(cand['logprob']):.1%}")
else:
    print("Skipped: this peek needs Ollama.")

'help'       18.6%
'gener'      16.2%
'time'       15.7%
'eff'        5.6%
'support'    5.3%


Two lessons in one output. First, the model is **not certain**: its favourite next token gets only a modest share; the rest is spread over alternatives. Second, look at the candidates: some are **pieces of words** (such as `gener`, the start of *generosity*). The model chooses token by token.

## 4. Temperature, by hand and in code

The model gives each candidate a raw score. **Softmax** turns scores into probabilities; **temperature** divides the scores first. These are the README numbers:

In [7]:
import numpy as np

candidates = ["order", "purchase", "patience", "banana"]
scores = np.array([3.0, 2.5, 1.5, -1.0])       # the model's raw scores (logits)

def softmax(z, temperature=1.0):
    e = np.exp(z / temperature)
    return e / e.sum()

for T in [0.5, 1.0, 2.0]:
    probs = softmax(scores, T)
    print(f"T={T}: " + "  ".join(f"{w} {p:.3f}" for w, p in zip(candidates, probs)))

T=0.5: order 0.705  purchase 0.259  patience 0.035  banana 0.000
T=1.0: order 0.541  purchase 0.328  patience 0.121  banana 0.010
T=2.0: order 0.419  purchase 0.326  patience 0.198  banana 0.057


Low temperature sharpens the distribution (the favourite gets 70.5%), high temperature flattens it (even *banana* gets 5.7%). **Top-p** (nucleus sampling) is a second knob: keep the smallest set of candidates whose probabilities add up to *p*, and sample only from those.

In [8]:
probs = softmax(scores, 1.0)
order = np.argsort(-probs)
cumulative = np.cumsum(probs[order])
keep = order[: np.searchsorted(cumulative, 0.9) + 1]
print("cumulative:", [f"{candidates[i]} {c:.3f}" for i, c in zip(order, cumulative)])
print("top-p 0.9 keeps:", [candidates[i] for i in keep])

cumulative: ['order 0.541', 'purchase 0.869', 'patience 0.990', 'banana 1.000']
top-p 0.9 keeps: ['order', 'purchase', 'patience']


Now on a real model. We ask for a product name five times at temperature 0 and five times at 1.2, and count how many different answers we get.

Temperature is passed to Ollama and to OpenAI-compatible providers. **Current Claude models do not accept sampling settings**, so the course helper ignores `temperature` when you use Claude; with Claude you would see variation even without it.

In [9]:
prompt = "Suggest one name for a new scented candle. Reply with the name only."
for T in [0.0, 1.2]:
    names = [ask(prompt, temperature=T, max_tokens=20).strip().strip('"*') for _ in range(5)]
    print(f"temperature {T}: {len(set(names))} different → {names}")

temperature 0.0: 1 different → ['Moonlit Garden', 'Moonlit Garden', 'Moonlit Garden', 'Moonlit Garden', 'Moonlit Garden']


temperature 1.2: 4 different → ['Midnight Blossom', 'Moonlit Lavender Dream', 'Moonlit Garden', 'Moonlit Garden', 'Moonlit Oasis']


At temperature 0 the model is (almost) deterministic: same prompt, same answer. At 1.2 it explores. Use low temperature for extraction, classification and support answers; higher for brainstorming.

## 5. The context window

Everything the model can "see" at once (instructions, documents, the conversation, its own answer) must fit in its **context window**, measured in tokens. How big are ShopWave's eleven policy documents?

In [10]:
doc_dir = repo_file("data/shopwave_docs/returns.md").parent
names = ["delivery", "international-shipping", "returns", "refunds", "damaged-items", "gift-wrap",
         "vouchers-and-promo-codes", "wholesale", "loyalty-stars", "contact-support", "christmas"]
total = sum(len(enc.encode(repo_file(f"data/shopwave_docs/{n}.md").read_text())) for n in names)
print(f"All 11 policy documents: about {total:,} tokens")
print(f"Course default context for Ollama: {int(course_llm.os.environ.get('LLM_CONTEXT', 8192)):,} tokens")

All 11 policy documents: about 1,752 tokens
Course default context for Ollama: 8,192 tokens


The policies fit comfortably. ShopWave's half a million transaction lines would not fit in any context window, and that is one reason the next lessons **retrieve** only the relevant pieces (09.10) instead of pasting everything in.

## Quiz answers

1. The next token, given all the tokens before it.
2. P(*red* | *is*) = 2/3: *is* is followed by *red* twice and *big* once.
3. Tokens. Pricing and context windows are counted in tokens, and rare words, codes and some languages use more tokens per word.
4. It divides the scores before softmax: low temperature makes the top choice more likely (more predictable), high temperature spreads probability out (more varied).
5. Pre-training on huge amounts of text (learn to predict the next token), instruction tuning on examples of good answers (learn to follow requests), then preference training such as RLHF (learn which answers people prefer).

**Next:** [09.8 Prompt engineering](../08-prompt-engineering/)